# Quantum-SAM: OpenEarthMap training

This clean Colab workflow trains **OpenEarthMap only**. Turn on a GPU in `Runtime → Change runtime type` before running. The dataset layout, pairing, RGB/indexed mask handling, and training parameters are all configured in `configs/openearthmap.yaml`.


In [ ]:
!git clone https://github.com/KAVINRAJ06/Quantum_CL.git
%cd Quantum_CL
!python -m pip install -q -U -r requirements-colab.txt
import torch
assert torch.cuda.is_available(), 'Enable a GPU: Runtime → Change runtime type → T4 GPU'
print('PyTorch:', torch.__version__, '| GPU:', torch.cuda.get_device_name(0))


## Kaggle token

Upload the `kaggle.json` token from your Kaggle settings. It stays only in this temporary Colab runtime.


In [ ]:
from google.colab import files
uploaded = files.upload()
assert 'kaggle.json' in uploaded, 'Please upload the Kaggle API token named kaggle.json'
!mkdir -p ~/.kaggle
!mv kaggle.json ~/.kaggle/kaggle.json
!chmod 600 ~/.kaggle/kaggle.json


In [ ]:
!mkdir -p raw/openearthmap data/openearthmap
!kaggle datasets download -d aletbm/global-land-cover-mapping-openearthmap -p raw/openearthmap --unzip
!python scripts/prepare_remote_sensing_data.py --source raw/openearthmap --destination data/openearthmap


In [ ]:
from pathlib import Path
from PIL import Image
for split in ('train', 'val'):
    images = list((Path('data/openearthmap/images') / split).glob('*'))
    masks = list((Path('data/openearthmap/masks') / split).glob('*'))
    assert images and masks, f'{split}: preparation found no pairs'
    print(f'{split}: {len(images)} images, {len(masks)} masks, first mask mode={Image.open(masks[0]).mode}')


In [ ]:
!python smoke_test.py
!python tests/test_data_pipeline.py
!python train.py --config configs/openearthmap.yaml
